# Ablation Evaluation

Three-step ablation evaluation for Simurgh's personalized RAG pipeline. Each step isolates the contribution of one trained component:

| Step | Retriever | Rewriter | Generator |
|------|-----------|----------|-----------|
| 1 | Frozen Qwen3-Embedding (no adapter) | None | gpt-5-nano |
| 2 | ROPG-KD Qwen3-Embedding (LoRA) | None | gpt-5-nano |
| 3 | ROPG-KD Qwen3-Embedding (LoRA) | DPO Qwen3-4B (LoRA) | gpt-5-nano |

Judge: `gemini-3.5-flash` scores each answer on persona alignment, pedagogical quality, faithfulness, overall (1–5) and accuracy (0/1).

**Kaggle setup checklist**
1. Enable GPU accelerator (T4 x1).
2. Enable internet access (models downloaded from Hugging Face; judge calls Gemini API).
3. Attach the `simurgh-data` dataset — must contain `data/chunks/corpus.jsonl`, `data/questions/`, `data/index/phase1.faiss` + `phase1_meta.json`, trained ROPG-KD and DPO adapters.
4. Set Kaggle secrets: `OPENAI_API_KEY`, `OPENAI_BASE_URL`, `GEMINI_API_KEY` (and optionally `GEMINI_ENDPOINT`).

**Colab setup checklist**
1. Set `RUNTIME = "colab"` in Cell 2 below.
2. Upload `simurgh-data/` to Google Drive at `MyDrive/simurgh-data/` — must contain `data/chunks/corpus.jsonl`, `data/questions/`, `ropg_kd_best/`, `dpo_best/`.
3. Add secrets via Colab Secrets (left sidebar → key icon): `OPENAI_API_KEY`, `OPENAI_BASE_URL`, `GEMINI_API_KEY`.
4. Enable GPU accelerator (T4 × 1 needed for embedder inference).
5. Results and index are saved to Google Drive under `MyDrive/simurgh-data/`.

In [ ]:
!pip uninstall -q -y torchao
!pip install -q "sentence-transformers==5.6.0" peft accelerate faiss-gpu litellm openai hazm unsloth

In [ ]:
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# ── Runtime selector ─────────────────────────────────────────────────────────
# Set RUNTIME to match where you are running this notebook.
RUNTIME = "kaggle"  # "kaggle" | "colab" | "local"
GDRIVE_BASE = "/content/drive/MyDrive/simurgh-data"  # Colab only

# ── Secrets ───────────────────────────────────────────────────────────────────
if RUNTIME == "kaggle":
    from kaggle_secrets import UserSecretsClient

    _secrets = UserSecretsClient()
    os.environ["OPENAI_API_KEY"] = _secrets.get_secret("OPENAI_API_KEY")
    os.environ["OPENAI_BASE_URL"] = _secrets.get_secret("OPENAI_BASE_URL")
    os.environ["GEMINI_API_KEY"] = _secrets.get_secret("GEMINI_API_KEY")
elif RUNTIME == "colab":
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    os.environ["OPENAI_BASE_URL"] = userdata.get("OPENAI_BASE_URL")
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
else:  # local
    pass  # read from environment / .env loaded externally

## Config

In [ ]:
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
if RUNTIME == "kaggle":
    DATASET_SLUG = "simurgh-data"
    DATA_ROOT = f"/kaggle/input/datasets/alirezahsn/{DATASET_SLUG}"
    WORKING = Path("/kaggle/working")
elif RUNTIME == "colab":
    DATA_ROOT = GDRIVE_BASE
    WORKING = Path(GDRIVE_BASE)
else:  # local
    DATA_ROOT = "."
    WORKING = Path(".")

RESULTS_DIR = WORKING / "results" / "ablation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    "seeds": [42, 43, 44],
    "test_persona": "newcomer",
    "corpus_jsonl": f"{DATA_ROOT}/data/chunks/corpus.jsonl",
    "questions_dir": f"{DATA_ROOT}/data/questions",
    "judge_model": "gemini/gemini-3.5-flash",
    "judge_max_retries": 3,
    "retrieval": {"top_k": 5},
    "steps": {
        "step1": {
            "name": "frozen_retriever_no_rewriter",
            "embedder_adapter_path": None,
            "rewriter_type": "none",
            "generator_model": "gpt-5-nano",
            "index_path": str(WORKING / "index" / "phase1.faiss"),
            "meta_path": str(WORKING / "index" / "phase1_meta.json"),
        },
        "step2": {
            "name": "ropg_kd_retriever_no_rewriter",
            "embedder_adapter_path": f"{DATA_ROOT}/ropg_kd_best",
            "rewriter_type": "none",
            "generator_model": "gpt-5-nano",
            "index_path": str(WORKING / "index" / "phase3.faiss"),
            "meta_path": str(WORKING / "index" / "phase3_meta.json"),
        },
        "step3": {
            "name": "ropg_kd_retriever_dpo_rewriter",
            "embedder_adapter_path": f"{DATA_ROOT}/ropg_kd_best",
            "rewriter_type": "dpo",
            "dpo_model": "Qwen/Qwen3-4B",
            "dpo_adapter_path": f"{DATA_ROOT}/dpo_best",
            "generator_model": "gpt-5-nano",
            "index_path": str(WORKING / "index" / "phase3.faiss"),
            "meta_path": str(WORKING / "index" / "phase3_meta.json"),
        },
    },
}

(WORKING / "index").mkdir(parents=True, exist_ok=True)
print("Config ready.")

## Core classes and helpers

In [ ]:
# ── Persian normalizer ───────────────────────────────────────────────────────
import re
import hazm

_normalizer = hazm.Normalizer()
_DIGIT_TABLE = str.maketrans(
    "".join(chr(i) for i in range(0x0660, 0x066A))
    + "".join(chr(i) for i in range(0x06F0, 0x06FA)),
    "0123456789" * 2,
)

def normalize(text):
    text = _normalizer.normalize(text)
    text = text.translate(_DIGIT_TABLE)
    return re.sub(r"\s+", " ", text).strip()


# ── Hit dataclass ────────────────────────────────────────────────────────────
from dataclasses import dataclass

@dataclass(frozen=True)
class Hit:
    raw_text: str
    source: str
    chunk_index: int
    score: float


# ── Qwen3 Embedder ───────────────────────────────────────────────────────────
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

class Qwen3Embedder:
    def __init__(self, model_name="Qwen/Qwen3-Embedding-0.6B", device="cuda",
                 batch_size=32, adapter_path=None):
        self.model = SentenceTransformer(
            model_name, device=device, trust_remote_code=True
        )
        if adapter_path is not None:
            from peft import PeftModel
            PeftModel.from_pretrained(
                self.model[0].auto_model, adapter_path
            ).merge_and_unload()
        self.batch_size = batch_size
        self.dim = self.model.get_embedding_dimension()

    def encode(self, texts):
        vecs = self.model.encode(
            texts, batch_size=self.batch_size,
            normalize_embeddings=True, show_progress_bar=False,
        )
        return np.array(vecs, dtype=np.float32)

    def encode_query(self, texts, instruction=""):
        kwargs = dict(batch_size=self.batch_size, normalize_embeddings=True,
                      show_progress_bar=False)
        if instruction:
            kwargs["prompt"] = f"Instruct: {instruction}\nQuery: "
        vecs = self.model.encode(texts, **kwargs)
        return np.array(vecs, dtype=np.float32)


# ── Dense FAISS store ────────────────────────────────────────────────────────
import faiss
import json

class DenseIndex:
    """Build or load a FAISS IndexFlatIP index from a pre-chunked corpus JSONL."""

    def __init__(self, index_path, meta_path, embedder):
        self.index_path = Path(index_path)
        self.meta_path = Path(meta_path)
        self.embedder = embedder
        self._meta = []
        if self.index_path.exists() and self.meta_path.exists():
            self._index = faiss.read_index(str(self.index_path))
            self._meta = json.loads(self.meta_path.read_text(encoding="utf-8"))
            print(f"Loaded existing index: {self._index.ntotal} vectors from {self.index_path}")
        else:
            self._index = faiss.IndexFlatIP(embedder.dim)

    def build(self, corpus_jsonl, force=False):
        if self._index.ntotal > 0 and not force:
            print(f"Index already built ({self._index.ntotal} vectors). Skipping.")
            return
        chunks = []
        with open(corpus_jsonl, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    chunks.append(json.loads(line))
        texts = [normalize(c["text"]) for c in chunks]
        print(f"Encoding {len(texts)} chunks...")
        vecs = self.embedder.encode(texts)
        self._index.add(vecs)
        self._meta = [
            {"raw_text": c["text"], "source": c.get("source", c.get("chunk_id", "")),
             "chunk_index": i}
            for i, c in enumerate(chunks)
        ]
        faiss.write_index(self._index, str(self.index_path))
        self.meta_path.write_text(
            json.dumps(self._meta, ensure_ascii=False), encoding="utf-8"
        )
        print(f"Index built: {self._index.ntotal} vectors → {self.index_path}")

    def search(self, query, top_k=5, instruction=""):
        if self._index.ntotal == 0:
            return []
        vec = self.embedder.encode_query([normalize(query)], instruction=instruction)
        k = min(top_k, self._index.ntotal)
        scores, indices = self._index.search(vec, k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0:
                continue
            m = self._meta[idx]
            results.append(Hit(
                raw_text=m["raw_text"], source=m["source"],
                chunk_index=m["chunk_index"], score=float(score),
            ))
        return results


# ── OpenAI-compatible LLM client ─────────────────────────────────────────────
import openai

class LLMClient:
    def __init__(self, model, temperature=0.2, max_tokens=800):
        self.client = openai.OpenAI(
            base_url=os.environ.get("OPENAI_BASE_URL"),
            api_key=os.environ.get("OPENAI_API_KEY", "local"),
        )
        self.model = model
        self.temperature = temperature
        self.max_tokens = max_tokens

    def chat(self, messages):
        resp = self.client.chat.completions.create(
            model=self.model, messages=messages,
            temperature=self.temperature,
            max_completion_tokens=self.max_tokens,
        )
        return resp.choices[0].message.content or ""


# ── MCQ-aware RAG prompt ─────────────────────────────────────────────────────
_MCQ_SYSTEM = (
    "You are a Persian-language educational assistant. "
    "Use ONLY the retrieved passages below to answer. "
    "Identify which multiple-choice option is correct and explain why. "
    "Explain why each other option is wrong, with evidence from the passages. "
    "Always write your answer in Persian (\u0641\u0627\u0631\u0633\u06cc), in fluent prose, "
    "and cite source numbers like [1], [2]."
)

def build_mcq_rag_prompt(stem, options, hits):
    if hits:
        context = "\n\n".join(
            f"[{i}] (source: {h.source}) {h.raw_text}"
            for i, h in enumerate(hits, start=1)
        )
    else:
        context = "(No documents found.)"
    options_text = "\n".join(f"{i + 1}. {opt}" for i, opt in enumerate(options))
    user = (
        f"Retrieved passages:\n{context}\n\n"
        f"{stem}\n\nOptions:\n{options_text}"
    )
    return [{"role": "system", "content": _MCQ_SYSTEM}, {"role": "user", "content": user}]


# ── Gemini judge ─────────────────────────────────────────────────────────────
import warnings
import litellm

_JUDGE_SYSTEM = (
    "You are an impartial evaluator for a Persian educational RAG system. "
    "You will be given a learner persona, an exam question with multiple-choice options, "
    "the correct answer, retrieved passages, and a generated explanation. "
    "Score the generated explanation on these dimensions:\n"
    "- persona_alignment (1-5): Does the depth, vocabulary, and tone match this learner's level and goal?\n"
    "- pedagogical_quality (1-5): Does the explanation help the learner understand? "
    "Is the correct option identified and explained clearly? Are wrong options ruled out with good reasoning?\n"
    "- faithfulness (1-5): Are all claims grounded in the retrieved passages? "
    "A fluent answer that contradicts or ignores the passages scores 1.\n"
    "- overall (1-5): Your holistic judgment of the response quality for this specific learner.\n"
    "- accuracy (0 or 1): Did the generated explanation correctly identify the right answer? 1 if yes, 0 if no or unclear.\n\n"
    "Respond with ONLY valid JSON, no markdown, no explanation:\n"
    '{"persona_alignment": <int>, "pedagogical_quality": <int>, "faithfulness": <int>, "overall": <int>, "accuracy": <int>}'
)
_JUDGE_SENTINEL = {"persona_alignment": -1, "pedagogical_quality": -1,
                   "faithfulness": -1, "overall": -1, "accuracy": -1}

class GeminiJudge:
    def __init__(self, model="gemini/gemini-3.5-flash", max_retries=3):
        self.model = model
        self.max_retries = max_retries
        litellm.api_key = os.environ.get("GEMINI_API_KEY", "")
        endpoint = os.environ.get("GEMINI_ENDPOINT")
        if endpoint:
            litellm.api_base = endpoint

    def score(self, persona, query, rewritten_query, options, correct_option, hits, answer):
        passages = "\n".join(
            f"[{i}] {h.raw_text[:300]}" for i, h in enumerate(hits, start=1)
        )
        opts = "\n".join(f"{i + 1}. {o}" for i, o in enumerate(options))
        user_msg = (
            f"Learner persona: {persona}\n\n"
            f"Question stem: {query}\n\n"
            f"Retrieval query used: {rewritten_query}\n\n"
            f"Options:\n{opts}\n\n"
            f"Correct answer: {correct_option}\n\n"
            f"Retrieved passages:\n{passages}\n\n"
            f"Generated explanation:\n{answer}"
        )
        messages = [{"role": "system", "content": _JUDGE_SYSTEM},
                    {"role": "user", "content": user_msg}]
        for attempt in range(self.max_retries):
            try:
                resp = litellm.completion(model=self.model, messages=messages)
                raw = resp.choices[0].message.content.strip()
                raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
                result = json.loads(raw)
                for k in _JUDGE_SENTINEL:
                    if k not in result:
                        result[k] = -1
                return result
            except (json.JSONDecodeError, KeyError, AttributeError) as e:
                if attempt == self.max_retries - 1:
                    warnings.warn(f"GeminiJudge: parse failed after {self.max_retries} attempts: {e}")
                    return _JUDGE_SENTINEL.copy()
        return _JUDGE_SENTINEL.copy()


# ── Personas ─────────────────────────────────────────────────────────────────
PERSONAS = {
    "crammer": (
        "A ninth-grader who finds the textbook hard to follow and has little background "
        "on this topic. Mainly wants to pass the exam — give the answer and what is needed "
        "to score — but it must be spelled out simply, step by step, with examples."
    ),
    "scholar": (
        "A ninth-grader who loves the subject and has read ahead. Wants the deeper literary "
        "or grammatical explanation — the 'why', historical context, and connections to other "
        "texts — not just the exam answer."
    ),
    "steady": (
        "A ninth-grader with average preparation who follows the textbook and does homework "
        "regularly. Needs a clear, organized explanation that matches what was taught in class, "
        "with the key points highlighted."
    ),
    "newcomer": (
        "A ninth-grader who recently transferred schools and missed several lessons. "
        "Has gaps in prerequisite knowledge. Needs foundational context before the direct "
        "answer, with simple language and no assumed prior knowledge."
    ),
}


# ── Question loader ──────────────────────────────────────────────────────────
def load_questions(questions_dir):
    questions = []
    for json_path in sorted(Path(questions_dir).glob("*.json")):
        with open(json_path, encoding="utf-8") as f:
            exam = json.load(f)
        passages = {p["group_id"]: p["text"] for p in exam.get("passages", [])}
        for q in exam.get("questions", []):
            if q.get("type") != "mcq":
                continue
            group_id = q.get("group_id")
            passage_text = passages.get(group_id) if group_id else None
            correct_idx = q["answer"] - 1  # 1-based → 0-based
            questions.append({
                "source_file": json_path.name,
                "exam_title": exam["exam_title"],
                "question_id": q["id"],
                "stem": q["stem"],
                "options": q["options"],
                "correct_answer_index": correct_idx,
                "correct_answer_text": q["options"][correct_idx],
                "group_id": group_id,
                "passage_text": passage_text,
            })
    return questions


def build_retrieval_query(q):
    if q["passage_text"]:
        return q["passage_text"] + "\n" + q["stem"]
    return q["stem"]


print("Core classes loaded.")

## Eval loop

In [ ]:
import random

def run_eval_step(step_name, index, embedder, llm, judge, questions, rewriter=None,
                  seed=42, top_k=5, test_persona="newcomer"):
    """Run one step's eval loop. Returns list of result dicts."""
    random.seed(seed)
    results = []
    for q in questions:
        retrieval_query = build_retrieval_query(q)
        for persona_id, persona_str in PERSONAS.items():
            if rewriter is not None:
                rewritten_query = rewriter.rewrite(persona_str, retrieval_query)
            else:
                rewritten_query = retrieval_query

            hits = index.search(rewritten_query, top_k=top_k, instruction=persona_str)
            messages = build_mcq_rag_prompt(q["stem"], q["options"], hits)
            answer = llm.chat(messages)

            scores = judge.score(
                persona=persona_str,
                query=q["stem"],
                rewritten_query=rewritten_query,
                options=q["options"],
                correct_option=q["correct_answer_text"],
                hits=hits,
                answer=answer,
            )

            record = {
                "seed": seed,
                "step": step_name,
                "persona_id": persona_id,
                "is_test_persona": persona_id == test_persona,
                "question_id": q["question_id"],
                "source_file": q["source_file"],
                "stem": q["stem"],
                "options": q["options"],
                "correct_answer_text": q["correct_answer_text"],
                "retrieval_query": retrieval_query,
                "rewritten_query": rewritten_query,
                "hit_sources": [h.source for h in hits],
                "answer": answer,
                **scores,
            }
            results.append(record)
            print(
                f"  {q['question_id']} | {persona_id:10s} | "
                f"overall={scores.get('overall', -1)} acc={scores.get('accuracy', -1)}"
            )
    return results


def save_results(results, step_key, seed):
    out = RESULTS_DIR / f"{step_key}_seed{seed}.jsonl"
    with open(out, "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"  → {len(results)} records → {out}")


# Shared objects
all_questions = load_questions(CFG["questions_dir"])
judge = GeminiJudge(model=CFG["judge_model"], max_retries=CFG["judge_max_retries"])
print(f"Loaded {len(all_questions)} MCQ questions.")

## Step 1 — Frozen retriever, no rewriter

In [ ]:
step1_cfg = CFG["steps"]["step1"]

embedder_frozen = Qwen3Embedder(
    adapter_path=step1_cfg["embedder_adapter_path"],  # None
    device="cuda",
)
index_step1 = DenseIndex(
    index_path=step1_cfg["index_path"],
    meta_path=step1_cfg["meta_path"],
    embedder=embedder_frozen,
)
index_step1.build(CFG["corpus_jsonl"])

llm_step1 = LLMClient(model=step1_cfg["generator_model"])

all_step1_results = []
for seed in CFG["seeds"]:
    print(f"\n=== Step 1 | seed={seed} ===")
    results = run_eval_step(
        step_name=step1_cfg["name"],
        index=index_step1,
        embedder=embedder_frozen,
        llm=llm_step1,
        judge=judge,
        questions=all_questions,
        rewriter=None,
        seed=seed,
        top_k=CFG["retrieval"]["top_k"],
        test_persona=CFG["test_persona"],
    )
    save_results(results, "step1", seed)
    all_step1_results.extend(results)

# Free GPU memory before loading next embedder
del embedder_frozen
torch.cuda.empty_cache()
print("\nStep 1 complete.")

## Step 2 — ROPG-KD retriever, no rewriter

In [ ]:
step2_cfg = CFG["steps"]["step2"]

embedder_ropg = Qwen3Embedder(
    adapter_path=step2_cfg["embedder_adapter_path"],
    device="cuda",
)
index_step2 = DenseIndex(
    index_path=step2_cfg["index_path"],
    meta_path=step2_cfg["meta_path"],
    embedder=embedder_ropg,
)
index_step2.build(CFG["corpus_jsonl"])  # rebuilds with ROPG-KD encoder

llm_step2 = LLMClient(model=step2_cfg["generator_model"])

all_step2_results = []
for seed in CFG["seeds"]:
    print(f"\n=== Step 2 | seed={seed} ===")
    results = run_eval_step(
        step_name=step2_cfg["name"],
        index=index_step2,
        embedder=embedder_ropg,
        llm=llm_step2,
        judge=judge,
        questions=all_questions,
        rewriter=None,
        seed=seed,
        top_k=CFG["retrieval"]["top_k"],
        test_persona=CFG["test_persona"],
    )
    save_results(results, "step2", seed)
    all_step2_results.extend(results)

print("\nStep 2 complete.")

## Step 3 — ROPG-KD retriever + DPO rewriter

Reuses the ROPG-KD index from Step 2. Loads the DPO-trained Qwen3-4B rewriter on top.

In [ ]:
# ── DPO Rewriter (inline) ─────────────────────────────────────────────────────
_REWRITE_SYSTEM = (
    "You are a query rewriting assistant for a Persian educational RAG system. "
    "Given a learner profile and an original question, rewrite the question as a "
    "retrieval query that will surface the most pedagogically useful passages for "
    "that specific learner. "
    "Rules: output ONLY the rewritten query — no explanation, no preamble, no quotes. "
    "Keep it in Persian if the original is Persian."
)

class DPORewriter:
    def __init__(self, model_name="Qwen/Qwen3-4B", adapter_path=None, max_new_tokens=200):
        from unsloth import FastLanguageModel
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name, load_in_4bit=True, max_seq_length=512,
        )
        if adapter_path is not None:
            from peft import PeftModel
            model = PeftModel.from_pretrained(model, adapter_path)
        FastLanguageModel.for_inference(model)
        self.model = model
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens

    def rewrite(self, profile_rendered, query):
        user = (
            f"Learner profile: {profile_rendered}\n\n"
            f"Original question: {query}\n\n"
            "Rewritten retrieval query:"
        )
        messages = [
            {"role": "system", "content": _REWRITE_SYSTEM},
            {"role": "user", "content": user},
        ]
        prompt_text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        inputs = self.tokenizer(prompt_text, return_tensors="pt").to(self.model.device)
        outputs = self.model.generate(
            **inputs, max_new_tokens=self.max_new_tokens, temperature=0.3, do_sample=True,
        )
        decoded = self.tokenizer.decode(outputs[0], skip_special_tokens=False)
        marker = "<|im_start|>assistant\n"
        if marker in decoded:
            decoded = decoded.rsplit(marker, 1)[-1]
        return decoded.strip()

In [ ]:
step3_cfg = CFG["steps"]["step3"]

dpo_rewriter = DPORewriter(
    model_name=step3_cfg["dpo_model"],
    adapter_path=step3_cfg["dpo_adapter_path"],
)

# Step 3 reuses index_step2 (same ROPG-KD encoder, same index)
llm_step3 = LLMClient(model=step3_cfg["generator_model"])

all_step3_results = []
for seed in CFG["seeds"]:
    print(f"\n=== Step 3 | seed={seed} ===")
    results = run_eval_step(
        step_name=step3_cfg["name"],
        index=index_step2,
        embedder=embedder_ropg,
        llm=llm_step3,
        judge=judge,
        questions=all_questions,
        rewriter=dpo_rewriter,
        seed=seed,
        top_k=CFG["retrieval"]["top_k"],
        test_persona=CFG["test_persona"],
    )
    save_results(results, "step3", seed)
    all_step3_results.extend(results)

print("\nStep 3 complete.")

## Results

In [ ]:
import csv

all_results = all_step1_results + all_step2_results + all_step3_results

step_order = [
    CFG["steps"]["step1"]["name"],
    CFG["steps"]["step2"]["name"],
    CFG["steps"]["step3"]["name"],
]
all_personas = list(PERSONAS.keys())
metrics = ["persona_alignment", "pedagogical_quality", "faithfulness", "overall", "accuracy"]

def _mean(subset, key):
    vals = [r.get(key, 0) or 0 for r in subset if r.get(key, -1) != -1]
    return round(sum(vals) / len(vals), 2) if vals else -1

rows = []
for step_name in step_order:
    for pid in all_personas:
        subset = [r for r in all_results if r["step"] == step_name and r["persona_id"] == pid]
        if not subset:
            continue
        row = {
            "step": step_name,
            "persona": pid,
            "is_test": pid == CFG["test_persona"],
            "n": len(subset),
        }
        for m in metrics:
            row[m] = _mean(subset, m)
        rows.append(row)

# Write summary CSV
summary_path = RESULTS_DIR / "summary.csv"
with open(summary_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)
print(f"Summary written to {summary_path}")

# Print table
header = f"{'step':<40} {'persona':<12} {'test':5} {'align':6} {'ped':6} {'faith':6} {'ovrl':6} {'acc':5} {'n':4}"
print("\n" + header)
print("-" * len(header))
for r in rows:
    print(
        f"{r['step']:<40} {r['persona']:<12} {str(r['is_test']):5} "
        f"{r['persona_alignment']:6} {r['pedagogical_quality']:6} "
        f"{r['faithfulness']:6} {r['overall']:6} {r['accuracy']:5} {r['n']:4}"
    )